In [ ]:
from dataclasses import dataclass
from openai import OpenAI
import os
import json

from pydantic import BaseModel
from typing import Literal


@dataclass(frozen=True)
class Provider:
    """One provider to reliably route requests across all inference providers."""

    name: str
    env_var: str
    is_free: bool
    base_url: str | None
    model: str

PROVIDERS = [
    
    Provider(
        "Groq",
        "GROQ_API_KEY",
        True,
        "https://api.groq.com/openai/v1",
        "openai/gpt-oss-120b"
    ),
    Provider(
        "OpenRouter",
        "OPENROUTER_API_KEY",
        True,
        "https://openrouter.ai/api/v1",
        "openai/gpt-sol-latest"
    ),
    Provider(
        "OpenAI",
        "OPENAI_API_KEY",
        True,
        None,
        "gpt-4o-mini"
    ),
]

def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider

    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(
        f"No provider key set, add one of {expected} "
        "to your environment variables"
    )

def build_client(provider: Provider) -> OpenAI:
    api_key = os.getenv(provider.env_var)

    if provider.base_url is None:
        return OpenAI(api_key=api_key)

    return OpenAI(
        api_key=api_key,
        base_url=provider.base_url,
    )

def have_any_key() -> bool:
    return any(os.getenv(p.env_var) for p in PROVIDERS)

print("Found a provider key." if have_any_key() else "No provider key found.")

def llm_reply(prompt: str) -> str:
    provider = select_provider()
    print(f"Using {provider.name} provider")
    client = build_client(provider)
    result = client.chat.completions.create(
        model=provider.model,
        max_tokens=500,
        messages=[{
            "role": "user",
            "content": prompt,
        }]
    )

    return result.choices[0].message.content or ""

In [ ]:
SYSTEM_PROMPT = """
You are a product-review sentiment analyst.

Read each review the user sends and classify its sentiment. We should also attach a confidence score for the sentiment
between 0 to 1 and a short explanation for the sentiment as a reason.

For each review, respond with ONLY a JSON object -  no other text or comments - in exactly the following shape:

{
    "sentiment": "positive" | "negative" | "neutral",
    "confidence": 0.0 to 1.0,
    "reason": "short explanation for the sentiment"
}

""".strip()

In [ ]:
def analyze_review(incoming_review: str)-> str:
    """send review to LLM ans ask it to classify the sentiment."""
    provider = select_provider()
    client = build_client(provider)

    result = client.chat.completions.create(
        model=provider.model,
        max_tokens=1000,
        messages=[
            {
                "role":"system",
                "content":SYSTEM_PROMPT,
            },
            {
                "role":"user",
                "content":incoming_review,
            }
        ],
    )

    return result.choices[0].message.content or ""


In [ ]:
REVIEWS = [
        "Absolutely love these headphones — great sound and the battery lasts all day!",
        "Arrived broken and customer service never replied. Very disappointed.",
        "It works. Nothing special, nothing terrible.",
    ]

In [ ]:
review1 =analyze_review(REVIEWS[0])
print(review1)

In [ ]:
review2 = analyze_review(REVIEWS[1])
print(review2)

In [ ]:
class SentimentResult(BaseModel):
    """Sentiment result from the LLM."""

    sentiment: Literal["positive", "negative", "neutral"]
    confidence: float
    reason: str

In [ ]:
review3 = analyze_review(REVIEWS[2])
print(review3)

In [ ]:
def validate_sentiment_reply(raw_reply: str) -> SentimentResult | str:
    try:
        parsed = json.loads(raw_reply)
        return SentimentResult(**parsed)
    except json.JSONDecodeError:
        return f"Invalid JSON: {raw_reply}"

In [ ]:
result = validate_sentiment_reply(review3)

In [ ]:
print(type(result))

In [ ]:
def route_by_sentiment(result: SentimentResult) -> str:
    """Route the review based on the sentiment."""
    if result.sentiment == "positive":
        return "Positive review"
    elif result.sentiment == "negative":
        return "Negative review, connect with customer service"
    else:
        return "Neutral review, no action needed"


if isinstance(result, SentimentResult):
    print(route_by_sentiment(result))
else:
    print(result)
